# 예측 결과(bbox) 시각화

`predict.py`가 만든 `outputs/predictions/*.csv`(annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score)를 읽어서, `data/raw/.../test_images/<image_id>.png` 위에 예측 bbox를 그려서 보여줍니다.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Rectangle
from PIL import Image

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from utils import load_config

CONFIG_PATH = PROJECT_ROOT / "configs" / "fasterrcnn_test.yaml"
config = load_config(str(CONFIG_PATH))

# predict.py가 실제로 저장하는 경로와 항상 같은 파일을 보도록 config에서 experiment_name을 읽는다.
PRED_CSV = PROJECT_ROOT / config["output"]["dir"] / "predictions" / f"{config['output']['experiment_name']}.csv"
TEST_IMAGES_DIR = PROJECT_ROOT / "data" / "raw" / "sprint_ai_project1_data" / "test_images"
CATEGORY_MAPPING_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "category_mapping.json"
SCORE_THRESHOLD = 0.0  # 이미 predict.py에서 걸러진 값이지만, 노트북에서 더 좁혀보고 싶으면 조정

print(f"PRED_CSV = {PRED_CSV}")

In [ ]:
# 알약 이름이 한글이라 matplotlib 기본 폰트(DejaVu Sans)로는 네모(□)로 깨져서 나온다.
# 시스템에 설치된 한글 폰트를 찾아서 적용한다 (mac: AppleGothic/Apple SD Gothic Neo,
# Colab/Linux: Nanum Gothic 등). 아무것도 없으면 설치 안내를 출력한다.
import matplotlib.font_manager as fm

KOREAN_FONT_CANDIDATES = [
    "AppleGothic",
    "Apple SD Gothic Neo",
    "Malgun Gothic",
    "NanumGothic",
    "Nanum Gothic",
    "Noto Sans CJK KR",
    "Noto Sans KR",
]
available = {f.name for f in fm.fontManager.ttflist}
korean_font = next((name for name in KOREAN_FONT_CANDIDATES if name in available), None)

if korean_font:
    plt.rcParams["font.family"] = korean_font
    plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트는 마이너스(-) 글자가 없는 경우가 많음
    print(f"한글 폰트 적용: {korean_font}")
else:
    print(
        "한글 폰트를 찾지 못했습니다. 이름이 네모(□)로 표시된다면 폰트를 설치하세요.\n"
        "  - Colab/Linux: !apt-get -y install fonts-nanum 후 런타임 재시작,\n"
        "    그 다음 matplotlib.font_manager._load_fontmanager(try_read_cache=False) 필요할 수 있음\n"
        "  - macOS: 기본 AppleGothic이 있어야 정상이라 이 메시지가 뜨면 matplotlib 폰트 캐시 문제일 수 있음"
    )

## 데이터 로드

In [ ]:
df = pd.read_csv(PRED_CSV)
df = df[df["score"] >= SCORE_THRESHOLD]
print(len(df), "개 예측,", df["image_id"].nunique(), "개 이미지")
df.head()

In [ ]:
# category_id(predict.py가 category_mapping.json으로 원본 category_id로 되돌려놓은 값)
# -> 알약 이름. 없으면 category_id를 그대로 보여준다.
category_id_to_name = {}
if CATEGORY_MAPPING_PATH.exists():
    with open(CATEGORY_MAPPING_PATH) as f:
        category_mapping = json.load(f)
    category_id_to_name = {entry["category_id"]: entry["name"] for entry in category_mapping}
else:
    print(f"{CATEGORY_MAPPING_PATH} 없음 — category_id를 이름 없이 표시합니다.")

## 시각화 함수

In [ ]:
def draw_predictions(image_id: int, ax=None):
    """image_id에 해당하는 test 이미지 위에 예측 bbox를 그려서 ax에 표시한다."""
    image_path = TEST_IMAGES_DIR / f"{image_id}.png"
    image = Image.open(image_path).convert("RGB")

    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))

    ax.imshow(image)
    ax.set_title(f"image_id={image_id}")
    ax.axis("off")

    rows = df[df["image_id"] == image_id]
    for _, row in rows.iterrows():
        category_id = int(row["category_id"])
        name = category_id_to_name.get(category_id, str(category_id))
        ax.add_patch(
            Rectangle(
                (row["bbox_x"], row["bbox_y"]),
                row["bbox_w"],
                row["bbox_h"],
                fill=False,
                edgecolor="red",
                linewidth=2,
            )
        )
        ax.text(
            row["bbox_x"],
            max(row["bbox_y"] - 5, 0),
            f"{name} ({row['score']:.2f})",
            color="white",
            fontsize=8,
            bbox={"facecolor": "red", "alpha": 0.7, "pad": 1},
        )
    return ax

## 이미지 하나만 확인

`IMAGE_ID`를 바꿔서 원하는 이미지를 확인하세요.

In [ ]:
IMAGE_ID = df["image_id"].iloc[0]
draw_predictions(IMAGE_ID)
plt.show()

## 여러 이미지 그리드로 확인

In [ ]:
N_COLS = 3
N_IMAGES = 9  # 앞에서부터 몇 개 볼지

image_ids = sorted(df["image_id"].unique())[:N_IMAGES]
n_rows = (len(image_ids) + N_COLS - 1) // N_COLS

fig, axes = plt.subplots(n_rows, N_COLS, figsize=(5 * N_COLS, 5 * n_rows))
for ax, image_id in zip(axes.flat, image_ids):
    draw_predictions(image_id, ax=ax)
for ax in axes.flat[len(image_ids):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## (선택) 슬라이더로 하나씩 넘겨보기

`ipywidgets`가 설치되어 있어야 동작합니다 (`pip install ipywidgets`).

In [ ]:
from ipywidgets import interact

all_image_ids = sorted(df["image_id"].unique())


@interact(index=(0, len(all_image_ids) - 1))
def browse(index=0):
    draw_predictions(all_image_ids[index])
    plt.show()